# Plot Backend Comparison: Matplotlib vs Bokeh

Side-by-side comparison of every plot type using both backends.

### Setup

In [ ]:
from __future__ import annotations

import base64
import io

import matplotlib.pyplot as plt
from bokeh.embed import file_html
from bokeh.resources import Resources
from IPython.display import HTML, display

from beamphysics import ParticleGroup
from beamphysics.plot_dispatch import get_backend

mpl_be = get_backend("mpl")
bokeh_be = get_backend("bokeh")

ROW_HEIGHT = 800


def mpl_to_base64(fig: plt.Figure) -> str:
    """Render a matplotlib figure to a base64-encoded PNG."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()


def bokeh_to_srcdoc(layout) -> str:
    """Render a bokeh layout to an HTML string suitable for iframe srcdoc."""
    html = file_html(layout, resources=Resources(mode="inline"), title="")
    return html.replace("&", "&amp;").replace('"', "&quot;")


def compare(title: str, mpl_fig, bokeh_layout):
    """Display a side-by-side comparison with fixed row height."""
    img_b64 = mpl_to_base64(mpl_fig)
    srcdoc = bokeh_to_srcdoc(bokeh_layout)
    display(
        HTML(f"""
        <h2 style="font-family: monospace">{title}</h2>
        <div style="display: flex; gap: 20px; align-items: flex-start; height: {ROW_HEIGHT}px;">
          <div style="flex: 1; min-width: 0; height: 100%; overflow: auto;">
            <h3 style="color: #666">Matplotlib</h3>
            <img src="data:image/png;base64,{img_b64}" style="max-width:100%" />
          </div>
          <div style="flex: 1; min-width: 0; height: 100%;">
            <h3 style="color: #666">Bokeh</h3>
            <iframe srcdoc="{srcdoc}" style="width:100%;height:calc(100% - 30px);border:none;"></iframe>
          </div>
        </div>
        """)
    )

In [ ]:
P = ParticleGroup("data/bmad_particles2.h5")
P.t = P.t - P["mean_t"]

## Density Plot (1D)

In [ ]:
compare(
    "P.plot('t')",
    mpl_be.density_plot(P, key="t"),
    bokeh_be.density_plot(P, key="t", show=False),
)

## Marginal Plot — x vs y

In [ ]:
compare(
    "P.plot('x', 'y')",
    mpl_be.marginal_plot(P, key1="x", key2="y"),
    bokeh_be.marginal_plot(P, key1="x", key2="y", show=False),
)

## Marginal Plot — x vs px with ellipse

In [ ]:
compare(
    "P.plot('x', 'px', ellipse=True)",
    mpl_be.marginal_plot(P, key1="x", key2="px", ellipse=True),
    bokeh_be.marginal_plot(P, key1="x", key2="px", ellipse=True, show=False),
)

## Marginal Plot — t vs energy

In [ ]:
compare(
    "P.plot('t', 'energy')",
    mpl_be.marginal_plot(P, key1="t", key2="energy"),
    bokeh_be.marginal_plot(P, key1="t", key2="energy", show=False),
)

## Marginal Plot — x vs y (spot)

In [ ]:
compare(
    "P.plot('x', 'y')",
    mpl_be.marginal_plot(P, key1="x", key2="y"),
    bokeh_be.marginal_plot(P, key1="x", key2="y", show=False),
)

## Slice Plot

In [ ]:
compare(
    "P.slice_plot('sigma_x', 'sigma_y')",
    mpl_be.slice_plot(P, "sigma_x", "sigma_y"),
    bokeh_be.slice_plot(P, "sigma_x", "sigma_y", show=False),
)

## Density + Slice Plot

In [ ]:
compare(
    "density_and_slice_plot(P, 't', 'energy', ...)",
    mpl_be.density_and_slice_plot(
        P, "t", "energy", stat_keys=["sigma_x", "sigma_y"], n_slice=200, bins=200
    ),
    bokeh_be.density_and_slice_plot(
        P,
        "t",
        "energy",
        stat_keys=["sigma_x", "sigma_y"],
        n_slice=200,
        bins=200,
        show=False,
    ),
)

# Wavefront Plots

In [ ]:
from beamphysics import Wavefront

W = Wavefront.from_gaussian(
    shape=(51, 51, 21),
    dx=10e-6,
    dy=10e-6,
    dz=10e-6,
    wavelength=1e-9,
    sigma0=50e-6,
    energy=1.0,
)
Wk = W.to_kspace()

## Wavefront Power

In [ ]:
compare(
    "W.plot_power()",
    W.plot_power(backend="mpl", return_figure=True),
    W.plot_power(backend="bokeh", return_figure=True, show=False),
)

## Wavefront Fluence

In [ ]:
compare(
    "W.plot_fluence()",
    W.plot_fluence(backend="mpl", return_figure=True),
    W.plot_fluence(backend="bokeh", return_figure=True, show=False),
)

## Wavefront Fluence with Marginals (plot2)

In [ ]:
compare(
    "W.plot2()",
    W.plot2(backend="mpl", return_figure=True),
    W.plot2(backend="bokeh", return_figure=True, show=False),
)

## Spectral Intensity (k-space)

In [ ]:
compare(
    "Wk.plot_spectral_intensity()",
    Wk.plot_spectral_intensity(backend="mpl", return_figure=True),
    Wk.plot_spectral_intensity(backend="bokeh", return_figure=True, show=False),
)

## Photon Energy Spectrum (k-space)

In [ ]:
compare(
    "Wk.plot_photon_energy_spectrum()",
    Wk.plot_photon_energy_spectrum(backend="mpl", return_figure=True),
    Wk.plot_photon_energy_spectrum(backend="bokeh", return_figure=True, show=False),
)